In [3]:
#!/usr/bin/env python3
"""
Combine predictions from 5 criteria and evaluate metrics
Usage: python combine_and_evaluate.py
"""

import pandas as pd
import numpy as np
import os
from pathlib import Path
import glob
from sklearn.metrics import cohen_kappa_score, mean_squared_error

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_DIR = "/home/user06/Interspeech_2026/Model/Model"
OUTPUT_DIR = os.path.join(BASE_DIR, "preds_final")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_SIZE = 1000   # Stratified sample size (None = use all)
RANDOM_SEED = 42

# Paths to individual prediction directories
PRED_DIRS = {
    'fluency': '/home/user06/Interspeech_2026/Model/Model/Preds_fluency',
    'pronunciation': '/home/user06/Interspeech_2026/Model/Model/Preds_pronunciation', 
    'grammar': '/home/user06/Interspeech_2026/Model/Model/Preds_grammar',
    'vocabulary': '/home/user06/Interspeech_2026/Model/Model/Preds_vocabulary',
    'content': '/home/user06/Interspeech_2026/Model/Model/Preds_content'
}

# Prediction file name patterns to try
PRED_FILE_PATTERNS = ["test_predictions.csv", "predictions.csv", "results.csv", "*.csv"]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_prediction_file(directory):
    """Find the prediction CSV file in directory"""
    if not os.path.exists(directory):
        return None
    
    for pattern in PRED_FILE_PATTERNS:
        matches = glob.glob(os.path.join(directory, pattern))
        if matches:
            return matches[0]  # Return first match
    return None

def round_half(score):
    """Round to nearest 0.5"""
    return np.round(score * 2) / 2

def round_to_int(score):
    """Round to nearest integer"""
    return np.round(score).astype(int)

def stratified_sample(df, score_col, n, seed=42):
    """
    Stratified sample of n rows, proportional to score distribution.
    Ensures full score range is represented.
    """
    df = df.copy()
    df['_bin'] = np.round(df[score_col]).astype(int)
    total = len(df)
    sampled = (
        df.groupby('_bin', group_keys=False)
        .apply(lambda g: g.sample(
            n=max(1, round(n * len(g) / total)),
            random_state=seed
        ))
        .head(n)
        .reset_index(drop=True)
    )
    sampled = sampled.drop(columns=['_bin'])
    return sampled

# ============================================================================
# LOAD PREDICTIONS
# ============================================================================

print("="*80)
print("COMBINING PREDICTIONS AND EVALUATING METRICS")
print("="*80)

# First, check what files exist
print("\n🔍 Searching for prediction files...")
for criterion, pred_dir in PRED_DIRS.items():
    print(f"\nChecking {criterion}: {pred_dir}")
    if os.path.exists(pred_dir):
        csv_files = glob.glob(os.path.join(pred_dir, "*.csv"))
        if csv_files:
            print(f"  ✓ Found {len(csv_files)} CSV file(s):")
            for f in csv_files:
                print(f"    - {os.path.basename(f)}")
        else:
            print(f"  ❌ No CSV files found")
    else:
        print(f"  ❌ Directory does not exist")

print("\n" + "="*80)
print("LOADING PREDICTIONS")
print("="*80)

# Dictionary to store dataframes
dfs = {}

for criterion, pred_dir in PRED_DIRS.items():
    pred_file = find_prediction_file(pred_dir)
    
    if pred_file is None:
        print(f"⚠️  WARNING: No prediction file found for {criterion} in {pred_dir}")
        continue
    
    try:
        df = pd.read_csv(pred_file)   # Load full CSV — will stratify-sample after merge
        print(f"✓ Loaded {criterion}: {len(df)} samples from {os.path.basename(pred_file)}")
        
        # Show first few column names for debugging
        print(f"  Columns: {df.columns.tolist()}")
        
        # Rename prediction column to criterion name
        # PRIORITIZE pred_score_raw for raw predictions before rounding
        pred_col_found = False
        for col in ['pred_score_raw', 'pred_score_rounded', 'prediction', 'pred', 'predicted', 'score', 'predicted_score']:
            if col in df.columns:
                df = df.rename(columns={col: f'pred_{criterion}'})
                pred_col_found = True
                print(f"  ✓ Using column '{col}' as prediction")
                break
        
        if not pred_col_found:
            print(f"  ⚠️  WARNING: Could not find prediction column, using first numeric column")
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) > 0:
                df = df.rename(columns={numeric_cols[0]: f'pred_{criterion}'})
        
        # Rename ground truth column if exists
        gt_col_found = False
        for col in ['true_score', 'ground_truth', 'label', 'gt', 'true', 'true_label']:
            if col in df.columns:
                df = df.rename(columns={col: f'gt_{criterion}'})
                gt_col_found = True
                print(f"  ✓ Using column '{col}' as ground truth")
                break
        
        if not gt_col_found:
            print(f"  ⚠️  WARNING: No ground truth column found for {criterion}")
        
        dfs[criterion] = df
        
    except Exception as e:
        print(f"❌ ERROR loading {criterion}: {str(e)}")

if len(dfs) == 0:
    print("\n❌ ERROR: No prediction files found!")
    print("\nPlease check:")
    print("1. Have you run inference for all 5 criteria?")
    print("2. Are the output directories correct?")
    print("3. What are the actual CSV file names?")
    raise SystemExit("No predictions to process")

print(f"\n✓ Successfully loaded {len(dfs)}/5 criteria")

# ============================================================================
# MERGE DATAFRAMES
# ============================================================================

print("\n" + "="*80)
print("MERGING PREDICTIONS")
print("="*80)

# Get base dataframe (first criterion)
first_criterion = list(dfs.keys())[0]
merged_df = dfs[first_criterion].copy()

# Determine merge key (adjust based on your CSV structure)
merge_key = None
for possible_key in ['Candidate_ID', 'sample_id', 'audio_path', 'id', 'file_path']:
    if possible_key in merged_df.columns:
        merge_key = possible_key
        print(f"✓ Using '{merge_key}' as merge key")
        break

if merge_key is None:
    print("⚠️  WARNING: No common ID column found, merging by index")

# Merge other criteria
for criterion, df in dfs.items():
    if criterion == first_criterion:
        continue
    
    if merge_key and merge_key in df.columns:
        # Keep only ID and prediction/ground truth columns
        cols_to_merge = [merge_key, f'pred_{criterion}']
        if f'gt_{criterion}' in df.columns:
            cols_to_merge.append(f'gt_{criterion}')
        cols_to_merge = [c for c in cols_to_merge if c in df.columns]
        
        merged_df = merged_df.merge(df[cols_to_merge], on=merge_key, how='inner')
        print(f"✓ Merged {criterion} ({len(merged_df)} samples after merge)")
    else:
        # Merge by index
        merged_df[f'pred_{criterion}'] = df[f'pred_{criterion}']
        if f'gt_{criterion}' in df.columns:
            merged_df[f'gt_{criterion}'] = df[f'gt_{criterion}']
        print(f"✓ Merged {criterion} by index")

print(f"\n✓ Final merged dataframe shape: {merged_df.shape}")

# ============================================================================
# STRATIFIED SAMPLING — 1000 samples across full score range
# ============================================================================

print("\n" + "="*80)
print(f"STRATIFIED SAMPLING  (target={SAMPLE_SIZE}, seed={RANDOM_SEED})")
print("="*80)

gt_strat_col = f'gt_{first_criterion}'
if SAMPLE_SIZE and len(merged_df) > SAMPLE_SIZE and gt_strat_col in merged_df.columns:
    # Show GT distribution before sampling
    bins_before = np.round(merged_df[gt_strat_col]).astype(int).value_counts().sort_index()
    print(f"  Before: {len(merged_df)} samples  |  score range [{merged_df[gt_strat_col].min():.0f}, {merged_df[gt_strat_col].max():.0f}]")

    merged_df = stratified_sample(merged_df, gt_strat_col, SAMPLE_SIZE, RANDOM_SEED)

    bins_after = np.round(merged_df[gt_strat_col]).astype(int).value_counts().sort_index()
    print(f"  After : {len(merged_df)} samples  |  score range [{merged_df[gt_strat_col].min():.0f}, {merged_df[gt_strat_col].max():.0f}]")
    print(f"\n  Score bin distribution (score → count):")
    for score, cnt in bins_after.items():
        bar = "█" * int(cnt / max(bins_after) * 30)
        print(f"    {score:>3}  {bar:<30}  {cnt}")
else:
    print(f"  ℹ️  Using all {len(merged_df)} samples (no sampling needed)")

# ============================================================================
# ROUND INDIVIDUAL CRITERIA PREDICTIONS TO INTEGER
# ============================================================================

print("\n" + "="*80)
print("ROUNDING INDIVIDUAL CRITERIA PREDICTIONS TO INTEGER")
print("="*80)

# Round individual criterion predictions to integer
# GT is already integer (0-10 with step 1), no need to round
for criterion in dfs.keys():
    pred_col = f'pred_{criterion}'
    
    if pred_col in merged_df.columns:
        original_range = f"[{merged_df[pred_col].min():.2f}, {merged_df[pred_col].max():.2f}]"
        merged_df[pred_col] = round_to_int(merged_df[pred_col])
        rounded_range = f"[{merged_df[pred_col].min()}, {merged_df[pred_col].max()}]"
        print(f"✓ Rounded {pred_col}: {original_range} → {rounded_range}")

print("ℹ️  Ground truth scores are already integers (0-10 step 1), no rounding needed")

# ============================================================================
# CALCULATE AVERAGE PREDICTION (with 0.5 rounding)
# ============================================================================

print("\n" + "="*80)
print("CALCULATING AVERAGE SCORES")
print("="*80)

# Calculate average of 5 criteria predictions (now integers)
# Then round the average to nearest 0.5
pred_cols = [f'pred_{c}' for c in dfs.keys() if f'pred_{c}' in merged_df.columns]
pred_average_raw = merged_df[pred_cols].mean(axis=1)
merged_df['pred_average'] = round_half(pred_average_raw)

print(f"✓ Calculated average from {len(pred_cols)} criteria: {[c.replace('pred_', '') for c in pred_cols]}")
print(f"  Raw average range: [{pred_average_raw.min():.2f}, {pred_average_raw.max():.2f}]")
print(f"  Rounded average (0.5) range: [{merged_df['pred_average'].min():.2f}, {merged_df['pred_average'].max():.2f}]")

# ============================================================================
# EVALUATE METRICS - INDIVIDUAL CRITERIA
# ============================================================================

print("\n" + "="*80)
print("EVALUATION METRICS - INDIVIDUAL CRITERIA (INTEGER ROUNDING)")
print("="*80)
print(f"{'Criterion':<15} {'MAE':<10} {'RMSE':<10} {'Acc@1':<10} {'QWK':<10} {'Samples':<10}")
print("-"*80)

results = {}

for criterion in dfs.keys():
    pred_col = f'pred_{criterion}'
    gt_col = f'gt_{criterion}'
    
    if pred_col not in merged_df.columns or gt_col not in merged_df.columns:
        print(f"{criterion:<15} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'(no GT)':<10}")
        continue
    
    # Drop NaN values
    valid_mask = merged_df[[pred_col, gt_col]].notna().all(axis=1)
    y_true = merged_df.loc[valid_mask, gt_col].values
    y_pred = merged_df.loc[valid_mask, pred_col].values
    
    # Already rounded to integer, so no need to round again
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    acc1 = np.mean(np.abs(y_true - y_pred) <= 1.0) * 100
    
    try:
        qwk = cohen_kappa_score(y_true.astype(int), y_pred.astype(int), weights='quadratic')
    except Exception as e:
        qwk = np.nan
    
    results[criterion] = {
        'mae': mae,
        'rmse': rmse,
        'acc1': acc1,
        'qwk': qwk,
        'n_samples': len(y_true)
    }
    
    print(f"{criterion.capitalize():<15} {mae:<10.4f} {rmse:<10.4f} {acc1:<9.2f}% {qwk:<10.4f} {len(y_true):<10}")

# ============================================================================
# EVALUATE METRICS - AVERAGE vs FINAL
# ============================================================================

print("\n" + "="*80)
print("EVALUATION METRICS - AVERAGE (5 CRITERIA) vs FINAL (0.5 ROUNDING)")
print("="*80)

# Try to find ground truth final score
gt_final_col = None
for col in ['gt_final', 'final', 'ground_truth_final', 'label_final', 'gt_overall', 'overall']:
    if col in merged_df.columns:
        gt_final_col = col
        print(f"✓ Found ground truth final scores in column: '{gt_final_col}'")
        break

if gt_final_col is None:
    # Compute average of individual GT (already integers 0-10), then round to 0.5
    gt_cols = [f'gt_{c}' for c in dfs.keys() if f'gt_{c}' in merged_df.columns]
    if len(gt_cols) >= 3:  # At least 3 criteria
        gt_average_raw = merged_df[gt_cols].mean(axis=1)
        merged_df['gt_final'] = round_half(gt_average_raw)
        gt_final_col = 'gt_final'
        print(f"ℹ️  Computed gt_final as rounded (0.5) average of {len(gt_cols)} integer GT criteria: {[c.replace('gt_', '') for c in gt_cols]}")
        print(f"  Raw GT average range: [{gt_average_raw.min():.2f}, {gt_average_raw.max():.2f}]")
        print(f"  Rounded GT final (0.5) range: [{merged_df['gt_final'].min():.2f}, {merged_df['gt_final'].max():.2f}]")
    else:
        print("❌ Cannot find or compute ground truth 'final' scores!")
        print(f"   Available GT columns: {[c for c in merged_df.columns if c.startswith('gt_')]}")
        print(f"   Note: Individual criterion evaluations are shown above")

if gt_final_col:
    valid_mask = merged_df[['pred_average', gt_final_col]].notna().all(axis=1)
    y_true_final = merged_df.loc[valid_mask, gt_final_col].values
    y_pred_final = merged_df.loc[valid_mask, 'pred_average'].values
    
    # Already rounded to 0.5, no need to round again
    mae_final = np.mean(np.abs(y_true_final - y_pred_final))
    rmse_final = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
    acc1_final = np.mean(np.abs(y_true_final - y_pred_final) <= 1.0) * 100
    
    try:
        # Convert 0.5-increments to integers by multiplying by 2
        y_true_int = (y_true_final * 2).astype(int)
        y_pred_int = (y_pred_final * 2).astype(int)
        qwk_final = cohen_kappa_score(y_true_int, y_pred_int, weights='quadratic')
    except Exception as e:
        print(f"  ⚠️  Warning: QWK calculation failed for final: {e}")
        qwk_final = np.nan
    
    results['average_vs_final'] = {
        'mae': mae_final,
        'rmse': rmse_final,
        'acc1': acc1_final,
        'qwk': qwk_final,
        'n_samples': len(y_true_final)
    }
    
    print(f"\n{'Metric':<20} {'Value':<15}")
    print("-"*40)
    print(f"{'MAE':<20} {mae_final:<15.4f}")
    print(f"{'RMSE':<20} {rmse_final:<15.4f}")
    print(f"{'Acc@1':<20} {acc1_final:<14.2f}%")
    print(f"{'QWK':<20} {qwk_final:<15.4f}")
    print(f"{'Samples':<20} {len(y_true_final):<15}")

# ============================================================================
# SAVE COMBINED PREDICTIONS
# ============================================================================

print("\n" + "="*80)
print("SAVING COMBINED PREDICTIONS")
print("="*80)

output_csv = os.path.join(OUTPUT_DIR, "combined_predictions.csv")
merged_df.to_csv(output_csv, index=False)
print(f"✓ Saved combined predictions to: {output_csv}")

# Save metrics summary
metrics_output = os.path.join(OUTPUT_DIR, "metrics_summary.txt")
with open(metrics_output, 'w') as f:
    f.write("="*80 + "\n")
    f.write("EVALUATION METRICS SUMMARY\n")
    f.write("="*80 + "\n\n")
    
    f.write("INDIVIDUAL CRITERIA (PREDICTIONS: INTEGER ROUNDING, GT: ALREADY INTEGER):\n")
    f.write(f"{'Criterion':<15} {'MAE':<10} {'RMSE':<10} {'Acc@1':<10} {'QWK':<10} {'Samples':<10}\n")
    f.write("-"*80 + "\n")
    for criterion, metrics in results.items():
        if criterion == 'average_vs_final':
            continue
        f.write(f"{criterion.capitalize():<15} {metrics['mae']:<10.4f} {metrics['rmse']:<10.4f} {metrics['acc1']:<9.2f}% {metrics['qwk']:<10.4f} {metrics['n_samples']:<10}\n")
    
    if 'average_vs_final' in results:
        f.write("\n" + "="*80 + "\n")
        f.write("AVERAGE (5 CRITERIA) vs FINAL (0.5 ROUNDING):\n")
        f.write("-"*80 + "\n")
        f.write(f"MAE:      {results['average_vs_final']['mae']:.4f}\n")
        f.write(f"RMSE:     {results['average_vs_final']['rmse']:.4f}\n")
        f.write(f"Acc@1:    {results['average_vs_final']['acc1']:.2f}%\n")
        f.write(f"QWK:      {results['average_vs_final']['qwk']:.4f}\n")
        f.write(f"Samples:  {results['average_vs_final']['n_samples']}\n")

print(f"✓ Saved metrics summary to: {metrics_output}")

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)


COMBINING PREDICTIONS AND EVALUATING METRICS

🔍 Searching for prediction files...

Checking fluency: /home/user06/Interspeech_2026/Model/Model/Preds_fluency
  ✓ Found 2 CSV file(s):
    - test_predictions.csv
    - test_predictions copy.csv

Checking pronunciation: /home/user06/Interspeech_2026/Model/Model/Preds_pronunciation
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

Checking grammar: /home/user06/Interspeech_2026/Model/Model/Preds_grammar
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

Checking vocabulary: /home/user06/Interspeech_2026/Model/Model/Preds_vocabulary
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

Checking content: /home/user06/Interspeech_2026/Model/Model/Preds_content
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

LOADING PREDICTIONS
✓ Loaded fluency: 2647 samples from test_predictions.csv
  Columns: ['Candidate_ID', 'true_score', 'pred_score_raw', 'pred_score_rounded', 'error', 'abs_error']
  ✓ Using column 'pred_score_raw' as prediction
  ✓

/tmp/ipykernel_1818009/1539249682.py:70: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(


In [ ]:
#!/usr/bin/env python3
"""
Combine predictions from 5 criteria and evaluate metrics
Usage: python combine_and_evaluate.py
"""

import pandas as pd
import numpy as np
import os
from pathlib import Path
import glob
from sklearn.metrics import cohen_kappa_score, mean_squared_error

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_DIR = "/home/user06/Interspeech_2026/Model/Model"
OUTPUT_DIR = os.path.join(BASE_DIR, "preds_final")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Paths to individual prediction directories
PRED_DIRS = {
    'fluency': '/home/user06/Interspeech_2026/Model/Model/Preds_fluency',
    'pronunciation': '/home/user06/Interspeech_2026/Model/Model/Preds_pronunciation', 
    'grammar': '/home/user06/Interspeech_2026/Model/Model/Preds_grammar',
    'vocabulary': '/home/user06/Interspeech_2026/Model/Model/Preds_vocabulary',
    'content': '/home/user06/Interspeech_2026/Model/Model/Preds_content'
}

# Prediction file name patterns to try
PRED_FILE_PATTERNS = ["test_predictions.csv", "predictions.csv", "results.csv", "*.csv"]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_prediction_file(directory):
    """Find the prediction CSV file in directory"""
    if not os.path.exists(directory):
        return None
    
    for pattern in PRED_FILE_PATTERNS:
        matches = glob.glob(os.path.join(directory, pattern))
        if matches:
            return matches[0]  # Return first match
    return None

def round_half(score):
    """Round to nearest 0.5"""
    return np.round(score * 2) / 2

def round_to_int(score):
    """Round to nearest integer"""
    return np.round(score).astype(int)

# ============================================================================
# LOAD PREDICTIONS
# ============================================================================

print("="*80)
print("COMBINING PREDICTIONS AND EVALUATING METRICS")
print("="*80)

# First, check what files exist
print("\n🔍 Searching for prediction files...")
for criterion, pred_dir in PRED_DIRS.items():
    print(f"\nChecking {criterion}: {pred_dir}")
    if os.path.exists(pred_dir):
        csv_files = glob.glob(os.path.join(pred_dir, "*.csv"))
        if csv_files:
            print(f"  ✓ Found {len(csv_files)} CSV file(s):")
            for f in csv_files:
                print(f"    - {os.path.basename(f)}")
        else:
            print(f"  ❌ No CSV files found")
    else:
        print(f"  ❌ Directory does not exist")

print("\n" + "="*80)
print("LOADING PREDICTIONS")
print("="*80)

# Dictionary to store dataframes
dfs = {}

for criterion, pred_dir in PRED_DIRS.items():
    pred_file = find_prediction_file(pred_dir)
    
    if pred_file is None:
        print(f"⚠️  WARNING: No prediction file found for {criterion} in {pred_dir}")
        continue
    
    try:
        df = pd.read_csv(pred_file)[:1000]  # Load only first 1000 rows for testing
        print(f"✓ Loaded {criterion}: {len(df)} samples from {os.path.basename(pred_file)}")
        
        # Show first few column names for debugging
        print(f"  Columns: {df.columns.tolist()}")
        
        # Rename prediction column to criterion name
        # PRIORITIZE pred_score_raw for raw predictions before rounding
        pred_col_found = False
        for col in ['pred_score_raw', 'pred_score_rounded', 'prediction', 'pred', 'predicted', 'score', 'predicted_score']:
            if col in df.columns:
                df = df.rename(columns={col: f'pred_{criterion}'})
                pred_col_found = True
                print(f"  ✓ Using column '{col}' as prediction")
                break
        
        if not pred_col_found:
            print(f"  ⚠️  WARNING: Could not find prediction column, using first numeric column")
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) > 0:
                df = df.rename(columns={numeric_cols[0]: f'pred_{criterion}'})
        
        # Rename ground truth column if exists
        gt_col_found = False
        for col in ['true_score', 'ground_truth', 'label', 'gt', 'true', 'true_label']:
            if col in df.columns:
                df = df.rename(columns={col: f'gt_{criterion}'})
                gt_col_found = True
                print(f"  ✓ Using column '{col}' as ground truth")
                break
        
        if not gt_col_found:
            print(f"  ⚠️  WARNING: No ground truth column found for {criterion}")
        
        dfs[criterion] = df
        
    except Exception as e:
        print(f"❌ ERROR loading {criterion}: {str(e)}")

if len(dfs) == 0:
    print("\n❌ ERROR: No prediction files found!")
    print("\nPlease check:")
    print("1. Have you run inference for all 5 criteria?")
    print("2. Are the output directories correct?")
    print("3. What are the actual CSV file names?")
    raise SystemExit("No predictions to process")

print(f"\n✓ Successfully loaded {len(dfs)}/5 criteria")

# ============================================================================
# MERGE DATAFRAMES
# ============================================================================

print("\n" + "="*80)
print("MERGING PREDICTIONS")
print("="*80)

# Get base dataframe (first criterion)
first_criterion = list(dfs.keys())[0]
merged_df = dfs[first_criterion].copy()

# Determine merge key (adjust based on your CSV structure)
merge_key = None
for possible_key in ['Candidate_ID', 'sample_id', 'audio_path', 'id', 'file_path']:
    if possible_key in merged_df.columns:
        merge_key = possible_key
        print(f"✓ Using '{merge_key}' as merge key")
        break

if merge_key is None:
    print("⚠️  WARNING: No common ID column found, merging by index")

# Merge other criteria
for criterion, df in dfs.items():
    if criterion == first_criterion:
        continue
    
    if merge_key and merge_key in df.columns:
        # Keep only ID and prediction/ground truth columns
        cols_to_merge = [merge_key, f'pred_{criterion}']
        if f'gt_{criterion}' in df.columns:
            cols_to_merge.append(f'gt_{criterion}')
        cols_to_merge = [c for c in cols_to_merge if c in df.columns]
        
        merged_df = merged_df.merge(df[cols_to_merge], on=merge_key, how='inner')
        print(f"✓ Merged {criterion} ({len(merged_df)} samples after merge)")
    else:
        # Merge by index
        merged_df[f'pred_{criterion}'] = df[f'pred_{criterion}']
        if f'gt_{criterion}' in df.columns:
            merged_df[f'gt_{criterion}'] = df[f'gt_{criterion}']
        print(f"✓ Merged {criterion} by index")

print(f"\n✓ Final merged dataframe shape: {merged_df.shape}")

# ============================================================================
# ROUND INDIVIDUAL CRITERIA PREDICTIONS TO INTEGER
# ============================================================================

print("\n" + "="*80)
print("ROUNDING INDIVIDUAL CRITERIA PREDICTIONS TO INTEGER")
print("="*80)

# Round individual criterion predictions to integer
# GT is already integer (0-10 with step 1), no need to round
for criterion in dfs.keys():
    pred_col = f'pred_{criterion}'
    
    if pred_col in merged_df.columns:
        original_range = f"[{merged_df[pred_col].min():.2f}, {merged_df[pred_col].max():.2f}]"
        merged_df[pred_col] = round_to_int(merged_df[pred_col])
        rounded_range = f"[{merged_df[pred_col].min()}, {merged_df[pred_col].max()}]"
        print(f"✓ Rounded {pred_col}: {original_range} → {rounded_range}")

print("ℹ️  Ground truth scores are already integers (0-10 step 1), no rounding needed")

# ============================================================================
# CALCULATE AVERAGE PREDICTION (with 0.5 rounding)
# ============================================================================

print("\n" + "="*80)
print("CALCULATING AVERAGE SCORES")
print("="*80)

# Calculate average of 5 criteria predictions (now integers)
# Then round the average to nearest 0.5
pred_cols = [f'pred_{c}' for c in dfs.keys() if f'pred_{c}' in merged_df.columns]
pred_average_raw = merged_df[pred_cols].mean(axis=1)
merged_df['pred_average'] = round_half(pred_average_raw)

print(f"✓ Calculated average from {len(pred_cols)} criteria: {[c.replace('pred_', '') for c in pred_cols]}")
print(f"  Raw average range: [{pred_average_raw.min():.2f}, {pred_average_raw.max():.2f}]")
print(f"  Rounded average (0.5) range: [{merged_df['pred_average'].min():.2f}, {merged_df['pred_average'].max():.2f}]")

# ============================================================================
# EVALUATE METRICS - INDIVIDUAL CRITERIA
# ============================================================================

print("\n" + "="*80)
print("EVALUATION METRICS - INDIVIDUAL CRITERIA (INTEGER ROUNDING)")
print("="*80)
print(f"{'Criterion':<15} {'MAE':<10} {'RMSE':<10} {'Acc@1':<10} {'QWK':<10} {'Samples':<10}")
print("-"*80)

results = {}

for criterion in dfs.keys():
    pred_col = f'pred_{criterion}'
    gt_col = f'gt_{criterion}'
    
    if pred_col not in merged_df.columns or gt_col not in merged_df.columns:
        print(f"{criterion:<15} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'(no GT)':<10}")
        continue
    
    # Drop NaN values
    valid_mask = merged_df[[pred_col, gt_col]].notna().all(axis=1)
    y_true = merged_df.loc[valid_mask, gt_col].values
    y_pred = merged_df.loc[valid_mask, pred_col].values
    
    # Already rounded to integer, so no need to round again
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    acc1 = np.mean(np.abs(y_true - y_pred) <= 1.0) * 100
    
    try:
        qwk = cohen_kappa_score(y_true.astype(int), y_pred.astype(int), weights='quadratic')
    except Exception as e:
        qwk = np.nan
    
    results[criterion] = {
        'mae': mae,
        'rmse': rmse,
        'acc1': acc1,
        'qwk': qwk,
        'n_samples': len(y_true)
    }
    
    print(f"{criterion.capitalize():<15} {mae:<10.4f} {rmse:<10.4f} {acc1:<9.2f}% {qwk:<10.4f} {len(y_true):<10}")

# ============================================================================
# EVALUATE METRICS - AVERAGE vs FINAL
# ============================================================================

print("\n" + "="*80)
print("EVALUATION METRICS - AVERAGE (5 CRITERIA) vs FINAL (0.5 ROUNDING)")
print("="*80)

# Try to find ground truth final score
gt_final_col = None
for col in ['gt_final', 'final', 'ground_truth_final', 'label_final', 'gt_overall', 'overall']:
    if col in merged_df.columns:
        gt_final_col = col
        print(f"✓ Found ground truth final scores in column: '{gt_final_col}'")
        break

if gt_final_col is None:
    # Compute average of individual GT (already integers 0-10), then round to 0.5
    gt_cols = [f'gt_{c}' for c in dfs.keys() if f'gt_{c}' in merged_df.columns]
    if len(gt_cols) >= 3:  # At least 3 criteria
        gt_average_raw = merged_df[gt_cols].mean(axis=1)
        merged_df['gt_final'] = round_half(gt_average_raw)
        gt_final_col = 'gt_final'
        print(f"ℹ️  Computed gt_final as rounded (0.5) average of {len(gt_cols)} integer GT criteria: {[c.replace('gt_', '') for c in gt_cols]}")
        print(f"  Raw GT average range: [{gt_average_raw.min():.2f}, {gt_average_raw.max():.2f}]")
        print(f"  Rounded GT final (0.5) range: [{merged_df['gt_final'].min():.2f}, {merged_df['gt_final'].max():.2f}]")
    else:
        print("❌ Cannot find or compute ground truth 'final' scores!")
        print(f"   Available GT columns: {[c for c in merged_df.columns if c.startswith('gt_')]}")
        print(f"   Note: Individual criterion evaluations are shown above")

if gt_final_col:
    valid_mask = merged_df[['pred_average', gt_final_col]].notna().all(axis=1)
    y_true_final = merged_df.loc[valid_mask, gt_final_col].values
    y_pred_final = merged_df.loc[valid_mask, 'pred_average'].values
    
    # Already rounded to 0.5, no need to round again
    mae_final = np.mean(np.abs(y_true_final - y_pred_final))
    rmse_final = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
    acc1_final = np.mean(np.abs(y_true_final - y_pred_final) <= 1.0) * 100
    
    try:
        # Convert 0.5-increments to integers by multiplying by 2
        y_true_int = (y_true_final * 2).astype(int)
        y_pred_int = (y_pred_final * 2).astype(int)
        qwk_final = cohen_kappa_score(y_true_int, y_pred_int, weights='quadratic')
    except Exception as e:
        print(f"  ⚠️  Warning: QWK calculation failed for final: {e}")
        qwk_final = np.nan
    
    results['average_vs_final'] = {
        'mae': mae_final,
        'rmse': rmse_final,
        'acc1': acc1_final,
        'qwk': qwk_final,
        'n_samples': len(y_true_final)
    }
    
    print(f"\n{'Metric':<20} {'Value':<15}")
    print("-"*40)
    print(f"{'MAE':<20} {mae_final:<15.4f}")
    print(f"{'RMSE':<20} {rmse_final:<15.4f}")
    print(f"{'Acc@1':<20} {acc1_final:<14.2f}%")
    print(f"{'QWK':<20} {qwk_final:<15.4f}")
    print(f"{'Samples':<20} {len(y_true_final):<15}")

# ============================================================================
# SAVE COMBINED PREDICTIONS
# ============================================================================

print("\n" + "="*80)
print("SAVING COMBINED PREDICTIONS")
print("="*80)

output_csv = os.path.join(OUTPUT_DIR, "combined_predictions.csv")
merged_df.to_csv(output_csv, index=False)
print(f"✓ Saved combined predictions to: {output_csv}")

# Save metrics summary
metrics_output = os.path.join(OUTPUT_DIR, "metrics_summary.txt")
with open(metrics_output, 'w') as f:
    f.write("="*80 + "\n")
    f.write("EVALUATION METRICS SUMMARY\n")
    f.write("="*80 + "\n\n")
    
    f.write("INDIVIDUAL CRITERIA (PREDICTIONS: INTEGER ROUNDING, GT: ALREADY INTEGER):\n")
    f.write(f"{'Criterion':<15} {'MAE':<10} {'RMSE':<10} {'Acc@1':<10} {'QWK':<10} {'Samples':<10}\n")
    f.write("-"*80 + "\n")
    for criterion, metrics in results.items():
        if criterion == 'average_vs_final':
            continue
        f.write(f"{criterion.capitalize():<15} {metrics['mae']:<10.4f} {metrics['rmse']:<10.4f} {metrics['acc1']:<9.2f}% {metrics['qwk']:<10.4f} {metrics['n_samples']:<10}\n")
    
    if 'average_vs_final' in results:
        f.write("\n" + "="*80 + "\n")
        f.write("AVERAGE (5 CRITERIA) vs FINAL (0.5 ROUNDING):\n")
        f.write("-"*80 + "\n")
        f.write(f"MAE:      {results['average_vs_final']['mae']:.4f}\n")
        f.write(f"RMSE:     {results['average_vs_final']['rmse']:.4f}\n")
        f.write(f"Acc@1:    {results['average_vs_final']['acc1']:.2f}%\n")
        f.write(f"QWK:      {results['average_vs_final']['qwk']:.4f}\n")
        f.write(f"Samples:  {results['average_vs_final']['n_samples']}\n")

print(f"✓ Saved metrics summary to: {metrics_output}")

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)


# 📊 Evaluation for OLD MODEL (Model_old)

Combine predictions from 5 criteria and evaluate metrics for the old model (Wav2Vec2 + Qwen2).


In [ ]:
#!/usr/bin/env python3
"""
Combine predictions from 5 criteria and evaluate metrics - OLD MODEL
Usage: Run this cell to evaluate model_old results
"""

import pandas as pd
import numpy as np
import os
from pathlib import Path
import glob
from sklearn.metrics import cohen_kappa_score, mean_squared_error

# ============================================================================
# CONFIGURATION - MODEL OLD
# ============================================================================

BASE_DIR = "/home/user06/Interspeech_2026/model_old"
OUTPUT_DIR = os.path.join(BASE_DIR, "preds_final")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Paths to individual prediction directories for OLD MODEL
PRED_DIRS = {
    'fluency': '/home/user06/Interspeech_2026/model_old/preds/preds_fluency',
    'pronunciation': '/home/user06/Interspeech_2026/model_old/preds/preds_pronunciation', 
    'grammar': '/home/user06/Interspeech_2026/model_old/preds/preds_grammar',
    'vocabulary': '/home/user06/Interspeech_2026/model_old/preds/preds_vocabulary',
    'content': '/home/user06/Interspeech_2026/model_old/preds/preds_content'
}

# Prediction file name patterns to try
PRED_FILE_PATTERNS = ["test_predictions.csv", "predictions.csv", "results.csv", "*.csv"]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_prediction_file(directory):
    """Find the prediction CSV file in directory"""
    if not os.path.exists(directory):
        return None
    
    for pattern in PRED_FILE_PATTERNS:
        matches = glob.glob(os.path.join(directory, pattern))
        if matches:
            return matches[0]  # Return first match
    return None

def round_half(score):
    """Round to nearest 0.5"""
    return np.round(score * 2) / 2

# ============================================================================
# LOAD PREDICTIONS
# ============================================================================

print("="*80)
print("COMBINING PREDICTIONS AND EVALUATING METRICS - OLD MODEL")
print("="*80)

# First, check what files exist
print("\n🔍 Searching for prediction files...")
for criterion, pred_dir in PRED_DIRS.items():
    print(f"\nChecking {criterion}: {pred_dir}")
    if os.path.exists(pred_dir):
        csv_files = glob.glob(os.path.join(pred_dir, "*.csv"))
        if csv_files:
            print(f"  ✓ Found {len(csv_files)} CSV file(s):")
            for f in csv_files:
                print(f"    - {os.path.basename(f)}")
        else:
            print(f"  ❌ No CSV files found")
    else:
        print(f"  ❌ Directory does not exist")

print("\n" + "="*80)
print("LOADING PREDICTIONS")
print("="*80)

# Dictionary to store dataframes
dfs = {}

for criterion, pred_dir in PRED_DIRS.items():
    pred_file = find_prediction_file(pred_dir)
    
    if pred_file is None:
        print(f"⚠️  WARNING: No prediction file found for {criterion} in {pred_dir}")
        continue
    
    try:
        df = pd.read_csv(pred_file)
        print(f"✓ Loaded {criterion}: {len(df)} samples from {os.path.basename(pred_file)}")
        
        # Show first few column names for debugging
        print(f"  Columns: {df.columns.tolist()}")
        
        # Rename prediction column to criterion name
        # Use pred_score_rounded (already rounded to 0.5)
        pred_col_found = False
        for col in ['pred_score_rounded', 'pred_score_raw', 'prediction', 'pred', 'predicted', 'score', 'predicted_score']:
            if col in df.columns:
                df = df.rename(columns={col: f'pred_{criterion}'})
                pred_col_found = True
                print(f"  ✓ Using column '{col}' as prediction")
                break
        
        if not pred_col_found:
            print(f"  ⚠️  WARNING: Could not find prediction column, using first numeric column")
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) > 0:
                df = df.rename(columns={numeric_cols[0]: f'pred_{criterion}'})
        
        # Rename ground truth column if exists
        gt_col_found = False
        for col in ['true_score', 'ground_truth', 'label', 'gt', 'true', 'true_label']:
            if col in df.columns:
                df = df.rename(columns={col: f'gt_{criterion}'})
                gt_col_found = True
                print(f"  ✓ Using column '{col}' as ground truth")
                break
        
        if not gt_col_found:
            print(f"  ⚠️  WARNING: No ground truth column found for {criterion}")
        
        dfs[criterion] = df
        
    except Exception as e:
        print(f"❌ ERROR loading {criterion}: {str(e)}")

if len(dfs) == 0:
    print("\n❌ ERROR: No prediction files found!")
    print("\nPlease check:")
    print("1. Have you run inference for all 5 criteria?")
    print("2. Are the output directories correct?")
    print("3. What are the actual CSV file names?")
    raise SystemExit("No predictions to process")

print(f"\n✓ Successfully loaded {len(dfs)}/5 criteria")

# ============================================================================
# MERGE DATAFRAMES
# ============================================================================

print("\n" + "="*80)
print("MERGING PREDICTIONS")
print("="*80)

# Get base dataframe (first criterion)
first_criterion = list(dfs.keys())[0]
merged_df = dfs[first_criterion].copy()

# Determine merge key (adjust based on your CSV structure)
merge_key = None
for possible_key in ['Candidate_ID', 'sample_id', 'audio_path', 'id', 'file_path']:
    if possible_key in merged_df.columns:
        merge_key = possible_key
        print(f"✓ Using '{merge_key}' as merge key")
        break

if merge_key is None:
    print("⚠️  WARNING: No common ID column found, merging by index")

# Merge other criteria
for criterion, df in dfs.items():
    if criterion == first_criterion:
        continue
    
    if merge_key and merge_key in df.columns:
        # Keep only ID and prediction/ground truth columns
        cols_to_merge = [merge_key, f'pred_{criterion}']
        if f'gt_{criterion}' in df.columns:
            cols_to_merge.append(f'gt_{criterion}')
        cols_to_merge = [c for c in cols_to_merge if c in df.columns]
        
        merged_df = merged_df.merge(df[cols_to_merge], on=merge_key, how='inner')
        print(f"✓ Merged {criterion} ({len(merged_df)} samples after merge)")
    else:
        # Merge by index
        merged_df[f'pred_{criterion}'] = df[f'pred_{criterion}']
        if f'gt_{criterion}' in df.columns:
            merged_df[f'gt_{criterion}'] = df[f'gt_{criterion}']
        print(f"✓ Merged {criterion} by index")

print(f"\n✓ Final merged dataframe shape: {merged_df.shape}")

# ============================================================================
# ENSURE PREDICTIONS ARE ROUNDED TO 0.5
# ============================================================================

print("\n" + "="*80)
print("ENSURING PREDICTIONS ARE ROUNDED TO 0.5")
print("="*80)

# Round individual criterion predictions to 0.5 (should already be rounded)
for criterion in dfs.keys():
    pred_col = f'pred_{criterion}'
    
    if pred_col in merged_df.columns:
        original_range = f"[{merged_df[pred_col].min():.2f}, {merged_df[pred_col].max():.2f}]"
        merged_df[pred_col] = round_half(merged_df[pred_col])
        rounded_range = f"[{merged_df[pred_col].min():.2f}, {merged_df[pred_col].max():.2f}]"
        print(f"✓ Rounded {pred_col}: {original_range} → {rounded_range}")

print("ℹ️  Ground truth scores are already rounded to 0.5 (0, 0.5, 1.0, ..., 10.0)")

# ============================================================================
# CALCULATE AVERAGE PREDICTION (with 0.5 rounding)
# ============================================================================

print("\n" + "="*80)
print("CALCULATING AVERAGE SCORES")
print("="*80)

# Calculate average of 5 criteria predictions (already rounded to 0.5)
# Then round the average to nearest 0.5 again
pred_cols = [f'pred_{c}' for c in dfs.keys() if f'pred_{c}' in merged_df.columns]
pred_average_raw = merged_df[pred_cols].mean(axis=1)
merged_df['pred_average'] = round_half(pred_average_raw)

print(f"✓ Calculated average from {len(pred_cols)} criteria: {[c.replace('pred_', '') for c in pred_cols]}")
print(f"  Raw average range: [{pred_average_raw.min():.2f}, {pred_average_raw.max():.2f}]")
print(f"  Rounded average (0.5) range: [{merged_df['pred_average'].min():.2f}, {merged_df['pred_average'].max():.2f}]")

# ============================================================================
# EVALUATE METRICS - INDIVIDUAL CRITERIA (0.5 ROUNDING)
# ============================================================================

print("\n" + "="*80)
print("EVALUATION METRICS - INDIVIDUAL CRITERIA (0.5 ROUNDING)")
print("="*80)
print(f"{'Criterion':<15} {'MAE':<10} {'RMSE':<10} {'Acc@0.5':<10} {'Acc@1':<10} {'QWK':<10} {'Samples':<10}")
print("-"*80)

results = {}

for criterion in dfs.keys():
    pred_col = f'pred_{criterion}'
    gt_col = f'gt_{criterion}'
    
    if pred_col not in merged_df.columns or gt_col not in merged_df.columns:
        print(f"{criterion:<15} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'(no GT)':<10}")
        continue
    
    # Drop NaN values
    valid_mask = merged_df[[pred_col, gt_col]].notna().all(axis=1)
    y_true = merged_df.loc[valid_mask, gt_col].values
    y_pred = merged_df.loc[valid_mask, pred_col].values
    
    # Metrics with 0.5 rounding
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    acc05 = np.mean(np.abs(y_true - y_pred) <= 0.5) * 100  # Exact match
    acc1 = np.mean(np.abs(y_true - y_pred) <= 1.0) * 100
    
    try:
        # Convert 0.5-increments to integers by multiplying by 2
        y_true_int = (y_true * 2).astype(int)
        y_pred_int = (y_pred * 2).astype(int)
        qwk = cohen_kappa_score(y_true_int, y_pred_int, weights='quadratic')
    except Exception as e:
        qwk = np.nan
    
    results[criterion] = {
        'mae': mae,
        'rmse': rmse,
        'acc05': acc05,
        'acc1': acc1,
        'qwk': qwk,
        'n_samples': len(y_true)
    }
    
    print(f"{criterion.capitalize():<15} {mae:<10.4f} {rmse:<10.4f} {acc05:<9.2f}% {acc1:<9.2f}% {qwk:<10.4f} {len(y_true):<10}")

# ============================================================================
# EVALUATE METRICS - AVERAGE vs FINAL (0.5 ROUNDING)
# ============================================================================

print("\n" + "="*80)
print("EVALUATION METRICS - AVERAGE (5 CRITERIA) vs FINAL (0.5 ROUNDING)")
print("="*80)

# Try to find ground truth final score
gt_final_col = None
for col in ['gt_final', 'final', 'ground_truth_final', 'label_final', 'gt_overall', 'overall']:
    if col in merged_df.columns:
        gt_final_col = col
        print(f"✓ Found ground truth final scores in column: '{gt_final_col}'")
        break

if gt_final_col is None:
    # Compute average of individual GT (already rounded to 0.5), then round to 0.5 again
    gt_cols = [f'gt_{c}' for c in dfs.keys() if f'gt_{c}' in merged_df.columns]
    if len(gt_cols) >= 3:  # At least 3 criteria
        gt_average_raw = merged_df[gt_cols].mean(axis=1)
        merged_df['gt_final'] = round_half(gt_average_raw)
        gt_final_col = 'gt_final'
        print(f"ℹ️  Computed gt_final as rounded (0.5) average of {len(gt_cols)} GT criteria: {[c.replace('gt_', '') for c in gt_cols]}")
        print(f"  Raw GT average range: [{gt_average_raw.min():.2f}, {gt_average_raw.max():.2f}]")
        print(f"  Rounded GT final (0.5) range: [{merged_df['gt_final'].min():.2f}, {merged_df['gt_final'].max():.2f}]")
    else:
        print("❌ Cannot find or compute ground truth 'final' scores!")
        print(f"   Available GT columns: {[c for c in merged_df.columns if c.startswith('gt_')]}")
        print(f"   Note: Individual criterion evaluations are shown above")

if gt_final_col:
    valid_mask = merged_df[['pred_average', gt_final_col]].notna().all(axis=1)
    y_true_final = merged_df.loc[valid_mask, gt_final_col].values
    y_pred_final = merged_df.loc[valid_mask, 'pred_average'].values
    
    # Already rounded to 0.5, no need to round again
    mae_final = np.mean(np.abs(y_true_final - y_pred_final))
    rmse_final = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
    acc05_final = np.mean(np.abs(y_true_final - y_pred_final) <= 0.5) * 100  # Exact match
    acc1_final = np.mean(np.abs(y_true_final - y_pred_final) <= 1.0) * 100
    
    try:
        # Convert 0.5-increments to integers by multiplying by 2
        y_true_int = (y_true_final * 2).astype(int)
        y_pred_int = (y_pred_final * 2).astype(int)
        qwk_final = cohen_kappa_score(y_true_int, y_pred_int, weights='quadratic')
    except Exception as e:
        print(f"  ⚠️  Warning: QWK calculation failed for final: {e}")
        qwk_final = np.nan
    
    results['average_vs_final'] = {
        'mae': mae_final,
        'rmse': rmse_final,
        'acc05': acc05_final,
        'acc1': acc1_final,
        'qwk': qwk_final,
        'n_samples': len(y_true_final)
    }
    
    print(f"\n{'Metric':<20} {'Value':<15}")
    print("-"*40)
    print(f"{'MAE':<20} {mae_final:<15.4f}")
    print(f"{'RMSE':<20} {rmse_final:<15.4f}")
    print(f"{'Acc@0.5 (exact)':<20} {acc05_final:<14.2f}%")
    print(f"{'Acc@1':<20} {acc1_final:<14.2f}%")
    print(f"{'QWK':<20} {qwk_final:<15.4f}")
    print(f"{'Samples':<20} {len(y_true_final):<15}")

# ============================================================================
# SAVE COMBINED PREDICTIONS
# ============================================================================

print("\n" + "="*80)
print("SAVING COMBINED PREDICTIONS")
print("="*80)

output_csv = os.path.join(OUTPUT_DIR, "combined_predictions_old_model.csv")
merged_df.to_csv(output_csv, index=False)
print(f"✓ Saved combined predictions to: {output_csv}")

# Save metrics summary
metrics_output = os.path.join(OUTPUT_DIR, "metrics_summary_old_model.txt")
with open(metrics_output, 'w') as f:
    f.write("="*80 + "\n")
    f.write("EVALUATION METRICS SUMMARY - OLD MODEL (Wav2Vec2 + Qwen2)\n")
    f.write("="*80 + "\n\n")
    
    f.write("INDIVIDUAL CRITERIA (0.5 ROUNDING):\n")
    f.write(f"{'Criterion':<15} {'MAE':<10} {'RMSE':<10} {'Acc@0.5':<10} {'Acc@1':<10} {'QWK':<10} {'Samples':<10}\n")
    f.write("-"*80 + "\n")
    for criterion, metrics in results.items():
        if criterion == 'average_vs_final':
            continue
        f.write(f"{criterion.capitalize():<15} {metrics['mae']:<10.4f} {metrics['rmse']:<10.4f} {metrics['acc05']:<9.2f}% {metrics['acc1']:<9.2f}% {metrics['qwk']:<10.4f} {metrics['n_samples']:<10}\n")
    
    if 'average_vs_final' in results:
        f.write("\n" + "="*80 + "\n")
        f.write("AVERAGE (5 CRITERIA) vs FINAL (0.5 ROUNDING):\n")
        f.write("-"*80 + "\n")
        f.write(f"MAE:         {results['average_vs_final']['mae']:.4f}\n")
        f.write(f"RMSE:        {results['average_vs_final']['rmse']:.4f}\n")
        f.write(f"Acc@0.5:     {results['average_vs_final']['acc05']:.2f}%\n")
        f.write(f"Acc@1:       {results['average_vs_final']['acc1']:.2f}%\n")
        f.write(f"QWK:         {results['average_vs_final']['qwk']:.4f}\n")
        f.write(f"Samples:     {results['average_vs_final']['n_samples']}\n")

print(f"✓ Saved metrics summary to: {metrics_output}")

print("\n" + "="*80)
print("EVALUATION COMPLETE - OLD MODEL!")
print("="*80)

# Display summary
if 'average_vs_final' in results:
    print("\n📊 SUMMARY (OLD MODEL - AVERAGE OF 5 CRITERIA):")
    print(f"  MAE:     {results['average_vs_final']['mae']:.4f}")
    print(f"  RMSE:    {results['average_vs_final']['rmse']:.4f}")
    print(f"  Acc@0.5: {results['average_vs_final']['acc05']:.2f}%")
    print(f"  Acc@1:   {results['average_vs_final']['acc1']:.2f}%")
    print(f"  QWK:     {results['average_vs_final']['qwk']:.4f}")


COMBINING PREDICTIONS AND EVALUATING METRICS - OLD MODEL

🔍 Searching for prediction files...

Checking fluency: /home/user06/Interspeech_2026/model_old/preds/preds_fluency
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

Checking pronunciation: /home/user06/Interspeech_2026/model_old/preds/preds_pronunciation
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

Checking grammar: /home/user06/Interspeech_2026/model_old/preds/preds_grammar
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

Checking vocabulary: /home/user06/Interspeech_2026/model_old/preds/preds_vocabulary
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

Checking content: /home/user06/Interspeech_2026/model_old/preds/preds_content
  ✓ Found 1 CSV file(s):
    - test_predictions.csv

LOADING PREDICTIONS
✓ Loaded fluency: 2647 samples from test_predictions.csv
  Columns: ['Candidate_ID', 'true_score', 'pred_score_raw', 'pred_score_rounded', 'error', 'abs_error']
  ✓ Using column 'pred_score_rounded' as prediction

# 📊 Score-Range Analysis: New vs Old Model — Fluency

Compare MAE in **Low / Mid / High** score bands, plus the distribution (% of samples) in ground truth vs predictions from each model.

| Band | Score range |
|------|-------------|
| Low  | ≤ 4.0       |
| Mid  | 4.0 < score ≤ 7.0 |
| High | > 7.0       |

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

# ============================================================================
# LOAD FLUENCY PREDICTIONS
# ============================================================================

NEW_FLUENCY = "/home/user06/Interspeech_2026/Model/Model/Preds_fluency/test_predictions.csv"
OLD_FLUENCY = "/home/user06/Interspeech_2026/model_old/preds/preds_fluency/test_predictions.csv"

df_new = pd.read_csv(NEW_FLUENCY)
df_old = pd.read_csv(OLD_FLUENCY)

# Standardise column names
# new model: true_score, pred_score_rounded
# old model: true_score, pred_score_rounded
y_true      = df_new["true_score"].values
pred_new    = df_new["pred_score_rounded"].values
pred_old    = df_old["pred_score_rounded"].values

print(f"Samples: {len(y_true)}  |  New model pred range: [{pred_new.min():.1f}, {pred_new.max():.1f}]  |  Old: [{pred_old.min():.1f}, {pred_old.max():.1f}]")

# ============================================================================
# SCORE BAND DEFINITIONS
# ============================================================================

BANDS = {
    "Low  (≤4.0)":  y_true <= 4.0,
    "Mid  (4–7)":   (y_true > 4.0) & (y_true <= 7.0),
    "High (>7.0)":  y_true > 7.0,
}

def band_mask_pred(pred, label):
    """Same band boundaries applied to predictions"""
    if "Low"  in label: return pred <= 4.0
    if "Mid"  in label: return (pred > 4.0) & (pred <= 7.0)
    if "High" in label: return pred > 7.0

def qwk(yt, yp):
    yt_i = np.round(yt * 2).astype(int)
    yp_i = np.round(yp * 2).astype(int)
    try:
        return cohen_kappa_score(yt_i, yp_i, weights="quadratic")
    except Exception:
        return float("nan")

# ============================================================================
# 1. MAE PER SCORE BAND — NEW vs OLD
# ============================================================================

print("\n" + "="*78)
print("  MAE PER SCORE BAND  —  Fluency (Criterion)")
print("="*78)
print(f"  {'Band':<14} {'N (GT)':>7}  {'MAE New':>9}  {'MAE Old':>9}  {'ΔMAE (new−old)':>16}  {'Better?':>8}")
print("  " + "─"*72)

mae_rows = []
for band_name, mask in BANDS.items():
    n = int(mask.sum())
    if n == 0:
        continue
    mae_n = float(np.abs(pred_new[mask] - y_true[mask]).mean())
    mae_o = float(np.abs(pred_old[mask] - y_true[mask]).mean())
    delta = mae_n - mae_o
    better = "✅ New" if delta < -0.005 else ("❌ Old" if delta > 0.005 else "≈ tie")
    print(f"  {band_name:<14} {n:>7}  {mae_n:>9.4f}  {mae_o:>9.4f}  {delta:>+16.4f}  {better:>8}")
    mae_rows.append(dict(band=band_name, n=n, mae_new=mae_n, mae_old=mae_o, delta=delta))

# Overall
mae_all_n = float(np.abs(pred_new - y_true).mean())
mae_all_o = float(np.abs(pred_old - y_true).mean())
delta_all  = mae_all_n - mae_all_o
better_all = "✅ New" if delta_all < -0.005 else ("❌ Old" if delta_all > 0.005 else "≈ tie")
print("  " + "─"*72)
print(f"  {'OVERALL':<14} {len(y_true):>7}  {mae_all_n:>9.4f}  {mae_all_o:>9.4f}  {delta_all:>+16.4f}  {better_all:>8}")

# ============================================================================
# 2. DISTRIBUTION (%) — GROUND TRUTH vs NEW PRED vs OLD PRED
# ============================================================================

print("\n" + "="*78)
print("  SCORE DISTRIBUTION (%)  —  GT  vs  New Model  vs  Old Model")
print("="*78)
print(f"  {'Band':<14} {'GT %':>8}  {'New Pred %':>12}  {'Old Pred %':>12}")
print("  " + "─"*54)

n_total = len(y_true)
for band_name, mask_gt in BANDS.items():
    pct_gt  = 100.0 * mask_gt.sum() / n_total
    pct_new = 100.0 * band_mask_pred(pred_new, band_name).sum() / n_total
    pct_old = 100.0 * band_mask_pred(pred_old, band_name).sum() / n_total
    print(f"  {band_name:<14} {pct_gt:>7.1f}%  {pct_new:>11.1f}%  {pct_old:>11.1f}%")

# ============================================================================
# 3. QWK PER SCORE BAND
# ============================================================================

print("\n" + "="*78)
print("  QWK PER SCORE BAND  —  Fluency")
print("="*78)
print(f"  {'Band':<14} {'N':>6}  {'QWK New':>9}  {'QWK Old':>9}")
print("  " + "─"*46)

for band_name, mask in BANDS.items():
    n = int(mask.sum())
    if n < 2:
        continue
    qwk_n = qwk(y_true[mask], pred_new[mask])
    qwk_o = qwk(y_true[mask], pred_old[mask])
    print(f"  {band_name:<14} {n:>6}  {qwk_n:>9.4f}  {qwk_o:>9.4f}")

qwk_all_n = qwk(y_true, pred_new)
qwk_all_o = qwk(y_true, pred_old)
print("  " + "─"*46)
print(f"  {'OVERALL':<14} {len(y_true):>6}  {qwk_all_n:>9.4f}  {qwk_all_o:>9.4f}")

# ============================================================================
# 4. SUMMARY TABLE (DataFrame)
# ============================================================================

print("\n" + "="*78)
print("  SUMMARY TABLE")
print("="*78)
rows = []
for band_name, mask in BANDS.items():
    n = int(mask.sum())
    if n == 0:
        continue
    rows.append({
        "Band":         band_name,
        "N (GT)":       n,
        "GT %":         f"{100*n/n_total:.1f}%",
        "New Pred %":   f"{100*band_mask_pred(pred_new, band_name).sum()/n_total:.1f}%",
        "Old Pred %":   f"{100*band_mask_pred(pred_old, band_name).sum()/n_total:.1f}%",
        "MAE New":      round(float(np.abs(pred_new[mask] - y_true[mask]).mean()), 4),
        "MAE Old":      round(float(np.abs(pred_old[mask] - y_true[mask]).mean()), 4),
        "ΔMAE":         round(float(np.abs(pred_new[mask]-y_true[mask]).mean()
                                    - np.abs(pred_old[mask]-y_true[mask]).mean()), 4),
        "QWK New":      round(qwk(y_true[mask], pred_new[mask]), 4),
        "QWK Old":      round(qwk(y_true[mask], pred_old[mask]), 4),
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))


Samples: 2647  |  New model pred range: [1.0, 9.0]  |  Old: [0.5, 8.5]

  MAE PER SCORE BAND  —  Fluency (Criterion)
  Band            N (GT)    MAE New    MAE Old    ΔMAE (new−old)   Better?
  ────────────────────────────────────────────────────────────────────────
  Low  (≤4.0)       1275     0.6275     0.7216           -0.0941     ✅ New
  Mid  (4–7)        1247     0.5389     0.7522           -0.2133     ✅ New
  High (>7.0)        125     1.0560     1.1360           -0.0800     ✅ New
  ────────────────────────────────────────────────────────────────────────
  OVERALL           2647     0.6060     0.7556           -0.1496     ✅ New

  SCORE DISTRIBUTION (%)  —  GT  vs  New Model  vs  Old Model
  Band               GT %    New Pred %    Old Pred %
  ──────────────────────────────────────────────────────
  Low  (≤4.0)       48.2%         42.8%         45.7%
  Mid  (4–7)        47.1%         54.4%         49.3%
  High (>7.0)        4.7%          2.8%          5.0%

  QWK PER SCORE BAND 